# CATAM 7.3 — Minimisation Methods

All algorithms live in `minimise.py`. This notebook drives the nine questions.

**Before running anything**: fill in the three gradient stubs (`grad_bedpan`, `grad_rosen`, `grad_quad3`) in `minimise.py`. The cell below verifies them against `numerical_gradient`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from minimise import (
    f_bedpan, grad_bedpan,
    f_rosen, grad_rosen,
    f_quad3, grad_quad3, HESS_QUAD3_INV,
    numerical_gradient,
    SteepestDescent, ConjugateGradient, DFP,
    minimise, print_table,
    plot_trajectory, plot_surface, plot_phi, line_search_auto,
)

rng = np.random.default_rng(0)
for f, g, dim in [(f_bedpan, grad_bedpan, 2), (f_rosen, grad_rosen, 2), (f_quad3, grad_quad3, 3)]:
    for _ in range(5):
        x = rng.uniform(-1.5, 1.5, size=dim)
        try:
            assert np.allclose(g(x), numerical_gradient(f, x), atol=1e-5), (f.__name__, x)
        except NotImplementedError:
            print(f'{f.__name__}: gradient stub not yet filled in')
            break
    else:
        print(f'{f.__name__}: analytic gradient matches numerical_gradient ✓')

## Q1 — Contour and surface plots; analytic minima

*(Work out minima analytically in the writeup. The plots below are visual aids.)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, f, title in [(axes[0], f_bedpan, 'f₄: bedpan'), (axes[1], f_rosen, 'f₅: Rosenbrock-like')]:
    xs = np.linspace(-1.5, 1.5, 200); ys = np.linspace(-1.5, 1.5, 200)
    X, Y = np.meshgrid(xs, ys)
    Z = np.vectorize(lambda a, b: f(np.array([a, b])))(X, Y)
    cs = ax.contour(X, Y, Z, levels=30)
    ax.clabel(cs, inline=True, fontsize=7)
    ax.set_title(title); ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

plot_surface(f_bedpan); plt.title('f₄'); plt.show()
plot_surface(f_rosen);  plt.title('f₅'); plt.show()

## Q2 — Steepest Descents on f₄, from (−1.0, −1.3), 10 iterations

In [ ]:
hist_q2 = minimise(SteepestDescent(), f_bedpan, grad_bedpan,
                   x0=np.array([-1.0, -1.3]), n_iter=10, lam_source='manual')
print_table(hist_q2)
plot_trajectory(f_bedpan, hist_q2); plt.title('Q2: Steepest Descents on f₄'); plt.show()

## Q3 — Steepest Descents on f₅, from (0.676, 0.443), 9–15 iterations

In [ ]:
hist_q3 = minimise(SteepestDescent(), f_rosen, grad_rosen,
                   x0=np.array([0.676, 0.443]), n_iter=12, lam_source='manual')
print_table(hist_q3)
plot_trajectory(f_rosen, hist_q3); plt.title('Q3: Steepest Descents on f₅'); plt.show()

## Q4 — Conjugate Gradients on f₄ (compare with Q2)

In [ ]:
hist_q4 = minimise(ConjugateGradient(), f_bedpan, grad_bedpan,
                   x0=np.array([-1.0, -1.3]), n_iter=10, lam_source='manual')
print_table(hist_q4)
plot_trajectory(f_bedpan, hist_q4); plt.title('Q4: CG on f₄'); plt.show()

## Q5 — Conjugate Gradients on f₅ (compare with Q3)

In [ ]:
hist_q5 = minimise(ConjugateGradient(), f_rosen, grad_rosen,
                   x0=np.array([0.676, 0.443]), n_iter=12, lam_source='manual')
print_table(hist_q5)
plot_trajectory(f_rosen, hist_q5); plt.title('Q5: CG on f₅'); plt.show()

## Q6 — DFP on f₆ (3-variable quadratic), preset λ*

Spec: 3 iterations from x₀ = (1, 1, 1) with λ* = 0.3942, 2.5522, 4.2202.
H should tend to the inverse Hessian in equation (10).

In [ ]:
hist_q6 = minimise(DFP(), f_quad3, grad_quad3,
                   x0=np.array([1.0, 1.0, 1.0]), n_iter=3,
                   lam_source=[0.3942, 2.5522, 4.2202])
print_table(hist_q6, show_H=True)
print('\nTarget G⁻¹:\n', HESS_QUAD3_INV)
print('\nDifference (final H − G⁻¹):\n', hist_q6[-1].H - HESS_QUAD3_INV)

# Invariant from the spec: H* @ p == q at each step.
for i in range(1, len(hist_q6)):
    p = hist_q6[i].g - hist_q6[i-1].g
    q = hist_q6[i].lam * hist_q6[i].s
    resid = hist_q6[i].H @ p - q
    print(f'step {i}: ‖H*p − q‖ = {np.linalg.norm(resid):.2e}')

In [ ]:
# Sensitivity: perturb each λ* by ±10% and compare the final f and H.
base = [0.3942, 2.5522, 4.2202]
for delta in [-0.1, -0.01, 0.01, 0.1]:
    lams = [l * (1 + delta) for l in base]
    h = minimise(DFP(), f_quad3, grad_quad3, x0=np.array([1.,1.,1.]),
                 n_iter=3, lam_source=lams, verbose=False)
    print(f'δ={delta:+.2f}  f_final={h[-1].f:.3e}  ‖H−G⁻¹‖={np.linalg.norm(h[-1].H - HESS_QUAD3_INV):.3e}')

## Q7 — DFP on f₄ (compare with Q2, Q4)

In [ ]:
hist_q7 = minimise(DFP(), f_bedpan, grad_bedpan,
                   x0=np.array([-1.0, -1.3]), n_iter=10, lam_source='manual')
print_table(hist_q7, show_H=True)
plot_trajectory(f_bedpan, hist_q7); plt.title('Q7: DFP on f₄'); plt.show()
print('\nFinal H:\n', hist_q7[-1].H)

## Q8 — DFP on f₅ (compare with Q3, Q5)

In [ ]:
hist_q8 = minimise(DFP(), f_rosen, grad_rosen,
                   x0=np.array([0.676, 0.443]), n_iter=12, lam_source='manual')
print_table(hist_q8, show_H=True)
plot_trajectory(f_rosen, hist_q8); plt.title('Q8: DFP on f₅'); plt.show()
print('\nFinal H:\n', hist_q8[-1].H)

## Q9 — Side-by-side comparison of the three methods

Re-runs everything with `lam_source='auto'` so the comparison does not depend on by-eye λ* choices.

In [ ]:
cases = [
    ('f₄', f_bedpan, grad_bedpan, np.array([-1.0, -1.3]), 10),
    ('f₅', f_rosen,  grad_rosen,  np.array([0.676, 0.443]), 12),
]
for name, f, g, x0, n in cases:
    fig, ax = plt.subplots(figsize=(7, 7))
    xs = np.linspace(-1.5, 1.5, 200); ys = np.linspace(-1.5, 1.5, 200)
    X, Y = np.meshgrid(xs, ys)
    Z = np.vectorize(lambda a, b: f(np.array([a, b])))(X, Y)
    ax.contour(X, Y, Z, levels=30, linewidths=0.5, colors='0.7')
    for algo_cls in (SteepestDescent, ConjugateGradient, DFP):
        h = minimise(algo_cls(), f, g, x0=x0, n_iter=n, lam_source='auto', verbose=False)
        plot_trajectory(f, h, ax=ax, label=algo_cls().name)
        print(f'{name} / {algo_cls().name}: f_final = {h[-1].f:.6g}')
    ax.set_title(f'{name}: all three methods'); plt.show()